# ATP 2-01.3 — Fine-Tuned LLM (Apple Silicon / MLX)

Fine-tune a local language model on ATP 2-01.3 (Intelligence Preparation of the Battlefield)
to serve as an on-device doctrine assistant for military intelligence questions.

**Hardware:** Mac with Apple Silicon (M1–M4)  
**Model:** Gemma 4 E4B (mlx-community/gemma-4-E4B-it-4bit)  
**Framework:** MLX + LoRA  

```
PDF → Chunks → Enrich → QA Pairs → Format → Train → Evaluate → Export GGUF
  1       2        3         4         5        6         7           8
```

**Prerequisites:**
- Mac with M-series chip, conda environment `caimll_finetuning` (Python 3.11)
- [Ollama](https://ollama.com) running with `gemma4:latest` pulled
- `ATP_2-01.3.pdf` in the project root
- `pip install mlx-lm` in the environment

**Kernel:** Select `caimll_finetuning` before running.

## Step 0 — Environment Check

In [ ]:
import subprocess, sys, shutil
from pathlib import Path

print(f"Python: {sys.version}")
print(f"mlx-lm : {'installed' if shutil.which('mlx_lm.lora') else 'MISSING — pip install mlx-lm'}")
print(f"Ollama : {'running' if subprocess.run(['ollama', 'list'], capture_output=True).returncode == 0 else 'NOT running — start Ollama'}")
print(f"PDF    : {'found' if Path('ATP_2-01.3.pdf').exists() else 'MISSING — copy ATP_2-01.3.pdf to project root'}")
print(f"scripts: {'OK' if Path('scripts/chunker.py').exists() else 'MISSING'}")

## Step 1 — Chunk PDF

Parses ATP 2-01.3 into paragraph-level chunks with chapter/appendix metadata.
Output: `data/chunks.jsonl`

In [ ]:
%run scripts/chunker.py --pdf ATP_2-01.3.pdf --out data/chunks.jsonl

In [ ]:
import json
chunks = [json.loads(l) for l in open('data/chunks.jsonl')]
print(f"Total chunks: {len(chunks)}")
print("\nSample chunk:")
print(json.dumps(chunks[0], indent=2))

## Step 2 — Enrich Chunks

Classifies each chunk by IPB step, content type, echelon, domain, and difficulty.
Output: `data/enriched.jsonl`

In [ ]:
%run scripts/enricher.py --chunks data/chunks.jsonl --out data/enriched.jsonl

In [ ]:
from collections import Counter
enriched = [json.loads(l) for l in open('data/enriched.jsonl')]
ipb_steps = Counter(e['metadata'].get('ipb_step') for e in enriched)
print(f"Total enriched: {len(enriched)}")
print("\nIPB step distribution:")
for step, cnt in sorted(ipb_steps.items()):
    label = {0:'Fundamentals',1:'Define OE',2:'Describe Effects',3:'Evaluate Threat',4:'Determine COAs'}.get(step, str(step))
    print(f"  Step {step} ({label}): {cnt}")

## Step 3 — Generate QA Pairs

Uses Ollama (gemma4:latest) to generate structured QA pairs with thinking traces.
Targets 5,000 pairs across all chapters. Run is resumable — safe to interrupt.

> **Note:** Stop Ollama BEFORE Step 5 (training) to free VRAM.

Output: `data/seeds.jsonl`

In [ ]:
%run scripts/generator.py --enriched data/enriched.jsonl --out data/seeds.jsonl --model gemma4:latest --target 5000

In [ ]:
from collections import Counter
seeds = [json.loads(l) for l in open('data/seeds.jsonl')]
qtypes = Counter(s.get('question_type') for s in seeds)
print(f"Total QA pairs: {len(seeds)}")
print("\nQuestion type distribution:")
for qt, cnt in sorted(qtypes.items(), key=lambda x: -x[1]):
    print(f"  {qt:<30}: {cnt}")
print("\nSample (question + thinking trace):")
s = seeds[0]
print(f"  Q: {s['question']}")
print(f"  Trace (first 200 chars): {s.get('thinking_trace','')[:200]}")
print(f"  A (first 200 chars): {s['answer'][:200]}")

## Step 4 — Format Training Data

Converts QA pairs into Gemma 4 chat-template format with thinking traces.
Splits 90/10 train/val.

Output: `data/train.jsonl`, `data/val.jsonl`

In [ ]:
%run scripts/formatter.py --seeds data/seeds.jsonl --train data/train.jsonl --val data/val.jsonl

In [ ]:
train = [json.loads(l) for l in open('data/train.jsonl')]
val   = [json.loads(l) for l in open('data/val.jsonl')]
print(f"Train: {len(train)} | Val: {len(val)}")
print("\nFormatted example (first 600 chars):")
print(train[0]['text'][:600])

## Step 5 — Train (MLX LoRA)

> **IMPORTANT:** Stop Ollama before running this cell to free unified memory.
> ```bash
> ollama stop
> ```

Trains Gemma 4 E4B with LoRA using MLX. Runs entirely on Apple Silicon.

Output: `outputs/atp-gemma4-e4b-mlx-v1/`

In [ ]:
import subprocess, sys
result = subprocess.run([
    sys.executable, "-m", "mlx_lm.lora",
    "--model",        "mlx-community/gemma-4-E4B-it-4bit",
    "--train",
    "--data",         "data",
    "--iters",        "2000",
    "--batch-size",   "4",
    "--lora-layers",  "16",
    "--learning-rate","2e-4",
    "--adapter-path", "outputs/atp-gemma4-e4b-mlx-v1",
    "--steps-per-eval","100",
    "--save-every",   "500",
], check=False)
print("Exit code:", result.returncode)

## Step 6 — Fuse Adapter

Merges LoRA adapter weights back into the base model.

Output: `outputs/atp-gemma4-e4b-mlx-v1/fused/`

In [ ]:
import subprocess, sys
result = subprocess.run([
    sys.executable, "-m", "mlx_lm.fuse",
    "--model",        "mlx-community/gemma-4-E4B-it-4bit",
    "--adapter-path", "outputs/atp-gemma4-e4b-mlx-v1",
    "--save-path",    "outputs/atp-gemma4-e4b-mlx-v1/fused",
], check=False)
print("Exit code:", result.returncode)

## Step 7 — Evaluate

Runs 24 evaluation questions against the fine-tuned model and scores keyword coverage.
Questions scoring < 0.5 are flagged for DPO.

Output: `eval/results/atp-gemma4-e4b-mlx-v1.json`

In [ ]:
%run scripts/eval.py --adapter outputs/atp-gemma4-e4b-mlx-v1 --out eval/results/atp-gemma4-e4b-mlx-v1.json --compare-base

In [ ]:
import json
from pathlib import Path

results_path = Path('eval/results/atp-gemma4-e4b-mlx-v1.json')
if results_path.exists():
    results = json.loads(results_path.read_text())
    summary = results.get('summary', {})
    print("=" * 50)
    print("  EVALUATION SUMMARY")
    print("=" * 50)
    print(f"  Fine-tuned avg score : {summary.get('ft_avg', 'N/A'):.4f}")
    if 'base_avg' in summary:
        print(f"  Base model avg score : {summary.get('base_avg', 'N/A'):.4f}")
        ft_avg   = summary.get('ft_avg', 0)
        base_avg = summary.get('base_avg', 1)
        if base_avg > 0:
            print(f"  Improvement          : +{(ft_avg - base_avg) / base_avg * 100:.1f}%")
    print(f"  DPO candidates       : {summary.get('dpo_candidates', 'N/A')}")
    print("=" * 50)
else:
    print("Results file not found — run eval cell first.")

## Step 7b — DPO (Optional)

Runs Direct Preference Optimization on questions that scored below 0.5 in eval.
Skip if eval scores are satisfactory.

Output: `outputs/atp-gemma4-e4b-mlx-v1-dpo/`

In [ ]:
# Only run if eval flagged weak questions
%run scripts/dpo.py \
    --sft   outputs/atp-gemma4-e4b-mlx-v1 \
    --eval  eval/results/atp-gemma4-e4b-mlx-v1.json \
    --seeds data/seeds.jsonl \
    --out   outputs/atp-gemma4-e4b-mlx-v1-dpo \
    --max-pairs 200

## Step 8 — Export GGUF

Converts fused model to GGUF format for deployment with Ollama.

Output: `burns/atp-gemma4-e4b-mlx-v1/`

In [ ]:
%run scripts/burn_gguf.py --model outputs/atp-gemma4-e4b-mlx-v1/fused --out burns/atp-gemma4-e4b-mlx-v1

## Step 9 — Deploy with Ollama

Register the GGUF model with Ollama and run it.

In [ ]:
import subprocess
from pathlib import Path

burns_dir = Path('burns/atp-gemma4-e4b-mlx-v1')

modelfile_content = """FROM ./model.Q4_K_M.gguf
SYSTEM \"You are a doctrine-grounded military intelligence assistant specializing in ATP 2-01.3 (Intelligence Preparation of the Battlefield, March 2019). Provide accurate, doctrinally grounded answers. Place citations at the END in [Reference: ATP 2-01.3, para X-Y] format.\"
"""

modelfile_path = burns_dir / 'Modelfile'
modelfile_path.write_text(modelfile_content)
print(f"Modelfile written to {modelfile_path}")
print("\nTo deploy:")
print(f"  cd {burns_dir}")
print("  ollama create atp-doctrine -f Modelfile")
print("  ollama run atp-doctrine")

## Summary

| Stage | Output | Status |
|-------|--------|--------|
| 1 — Chunk | `data/chunks.jsonl` | |
| 2 — Enrich | `data/enriched.jsonl` | |
| 3 — Generate | `data/seeds.jsonl` | |
| 4 — Format | `data/train.jsonl` + `data/val.jsonl` | |
| 5 — Train | `outputs/atp-gemma4-e4b-mlx-v1/` | |
| 6 — Fuse | `outputs/atp-gemma4-e4b-mlx-v1/fused/` | |
| 7 — Evaluate | `eval/results/atp-gemma4-e4b-mlx-v1.json` | |
| 8 — Export | `burns/atp-gemma4-e4b-mlx-v1/` | |

**Run from CLI instead:**
```bash
python scripts/run_pipeline.py --pdf ATP_2-01.3.pdf
```